In [5]:
from concurrent.futures import ThreadPoolExecutor, as_completed

import os
import openai

open_ai_keys = [
    os.getenv("OPENAI_KEY_1"),
    os.getenv("OPENAI_KEY_2"),
    os.getenv("OPENAI_KEY_3"),
    os.getenv("OPENAI_KEY_4"),
    os.getenv("OPENAI_KEY_5"),
    os.getenv("OPENAI_KEY_6"),
    os.getenv("OPENAI_KEY_7"),
    os.getenv("OPENAI_KEY_8"),
    os.getenv("OPENAI_KEY_9"),
    os.getenv("OPENAI_KEY_10"),
    os.getenv("OPENAI_KEY_11"),
    os.getenv("OPENAI_KEY_21"),
    os.getenv("OPENAI_KEY_13"),
    os.getenv("OPENAI_KEY_14"),
    os.getenv("OPENAI_KEY_15"),
    os.getenv("OPENAI_KEY_16"),
    os.getenv("OPENAI_KEY_17"),
    os.getenv("OPENAI_KEY_18"),
    os.getenv("OPENAI_KEY_19"),
    os.getenv("OPENAI_KEY_20")
]

openai.api_key = open_ai_keys[0]

executor = ThreadPoolExecutor(max_workers=10)

In [2]:
def prompt_standard_contextual(previous=None, current=None, next=None):
    if not current: # Ensure that there is a current sentence
        return "Error: A current sentence is required."

    # Initialize content with the current sentence
    content = f"Analyze this phrase (may be incomplete) for Ethos (E), Pathos (P), and Logos (L) from 0.0 - 9.9: '{current}'. Return a compact dictionary format like this: 'Ethos: x.x, Pathos: y.y, Logos: z.z, E_Explanation: text, P_Explanation: text, L_Explanation: text'. Explanations must be exactly 1 sentence long."
    
    if previous:
        content = f"Given the previous sentence: '{previous}', " + content
    if next:
        content = content + f" Also, consider the following sentence: '{next}'."

    prompt = openai.ChatCompletion.create(
        model="gpt-4",
        messages=[
            {
                "role": "system",
                "content": "You are a master debater and assistant to the judge of the US Presidential Debates."
            },
            {
                "role": "user",
                "content": content
            }
        ]
    )
    return prompt


In [3]:
import json

def process_lines(lines):
    """
    Processes the transcript lines into speaker turns.
    Each speaker turn starts with a line that has a colon ':'.
    """
    speaker_turns = []
    current_turn = []

    for line in lines:
        if ':' in line:  # This line is the start of a new speaker turn
            speaker_turns.append(' '.join(current_turn))  # Add the current turn to the turns list
            current_turn = [line]  # Start a new turn with this line
        else:
            current_turn.append(line)  # This line is part of the current turn

    speaker_turns.append(' '.join(current_turn))  # Add the last turn

    return speaker_turns[1:]  # The first element is empty, so we skip it


def get_turns(speaker_turns, i):
    """
    Returns the previous, current, and next turns given a list of speaker turns and an index.
    """
    prev_turn = speaker_turns[i - 1] if i > 0 else ''  # If this is the first turn, there is no previous turn
    curr_turn = speaker_turns[i]
    next_turn = speaker_turns[i + 1] if i < len(speaker_turns) - 1 else ''  # If this is the last turn, there is no next turn

    return prev_turn, curr_turn, next_turn


def read_transcript_and_analyze(input_file, output_file_full, output_file_content, startline=None, endline=None, startturn=None, endturn=None, runcount=None):
    # Error checking for parameters
    if startline is not None and startturn is not None:
        raise ValueError("Parameters 'startline' and 'startturn' are mutually exclusive. Use only 'startline' to specify the line number to start analysis, or 'startturn' to specify the speaker turn number to start analysis.")
    if endline is not None and endturn is not None:
        raise ValueError("Parameters 'endline' and 'endturn' are mutually exclusive. Use only 'endline' to specify the line number to end analysis, or 'endturn' to specify the speaker turn number to end analysis.")
    if startline is not None or endline is not None or startturn is not None or endturn is not None:
        if runcount is not None:
            raise ValueError("Parameter 'runcount' cannot be used together with line or turn parameters. Use 'runcount' alone or with either 'startline' and 'endline', or 'startturn' and 'endturn'.")
    
    with open(input_file, 'r') as file:
        lines = file.readlines()

    # Process lines and extract speaker turns
    speaker_turns = process_lines(lines)
    
    # Determine start and end points for analysis
    if startline is not None:
        start = startline
    elif startturn is not None:
        start = startturn
    else:
        start = 0

    if endline is not None:
        end = endline
    elif endturn is not None:
        end = endturn
    else:
        end = len(speaker_turns)  # Updated this line to consider turns instead of lines

    # Determine number of runs
    if runcount is not None:
        end = min(start + runcount, len(speaker_turns))  # Updated this line to consider turns instead of lines

    # Check if 'end' exceeds the length of the speaker turns
    if end > len(speaker_turns):  # Updated this line to consider turns instead of lines
        end = len(speaker_turns)  # Updated this line to consider turns instead of lines
        print(f"Warning: The end point exceeded the length of the speaker turns. It has been adjusted to the last speaker turn: {end}")

    for i in range(start, end):
        # Get previous, current and next turn
        prev_turn, curr_turn, next_turn = get_turns(speaker_turns, i)
        
        # Call OpenAI API and get response
        response = prompt_standard_contextual(prev_turn, curr_turn, next_turn)
        
        # Write complete response to JSON file
        with open(output_file_full, 'a') as outfile:
            json.dump(response, outfile)
            outfile.write('\n')  # Write newline character after each response
        
        # Extract 'content' field and write to second JSON file
        content = response['choices'][0]['message']['content']
        
        # Include the turns that were passed to GPT in the content
        content_with_turns = {
            "prev_turn": prev_turn,
            "curr_turn": curr_turn,
            "next_turn": next_turn,
            "content": content
        }
        
        with open(output_file_content, 'a') as outfile:
            json.dump(content_with_turns, outfile)  # Write the content with turns instead of just content
            outfile.write('\n')  # Write newline character after each response



In [4]:
read_transcript_and_analyze(input_file="1992-10-19-debate-preclean.txt", 
                            output_file_full="92-test-epl-test-turns_1_to_3.json", 
                            output_file_content="92-test-epl-run-content_1-to-3.json")

APIError: Request failed due to server shutdown {
  "error": {
    "message": "Request failed due to server shutdown",
    "type": "server_error",
    "param": null,
    "code": null
  }
}
 500 {'error': {'message': 'Request failed due to server shutdown', 'type': 'server_error', 'param': None, 'code': None}} {'Date': 'Fri, 09 Jun 2023 19:59:59 GMT', 'Content-Type': 'application/json', 'Content-Length': '141', 'Connection': 'keep-alive', 'access-control-allow-origin': '*', 'openai-model': 'gpt-4-0314', 'openai-organization': 'pulp', 'openai-processing-ms': '25117', 'openai-version': '2020-10-01', 'strict-transport-security': 'max-age=15724800; includeSubDomains', 'x-ratelimit-limit-requests': '200', 'x-ratelimit-limit-tokens': '40000', 'x-ratelimit-remaining-requests': '199', 'x-ratelimit-remaining-tokens': '39500', 'x-ratelimit-reset-requests': '300ms', 'x-ratelimit-reset-tokens': '750ms', 'x-request-id': '1b86cd1e7a47ef69896e68f030023230', 'CF-Cache-Status': 'DYNAMIC', 'Server': 'clo

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

def threaded_process(i, speaker_turns, output_file_full, output_file_content, lock1, lock2, key):
    # Get previous, current and next turn
    prev_turn, curr_turn, next_turn = get_turns(speaker_turns, i)
    
    # Set the API key
    openai.api_key = key
    
    # Call OpenAI API and get response
    response = prompt_standard_contextual(prev_turn, curr_turn, next_turn)
    
    # Write complete response to JSON file
    lock1.acquire()
    try:
        with open(output_file_full, 'a') as outfile:
            json.dump(response, outfile)
            outfile.write('\n')  # Write newline character after each response
    finally:
        lock1.release()

    # Extract 'content' field and write to second JSON file
    content = response['choices'][0]['message']['content']
    
    # Include the turns that were passed to GPT in the content
    content_with_turns = {
        "prev_turn": prev_turn,
        "curr_turn": curr_turn,
        "next_turn": next_turn,
        "content": content
    }
    
    lock2.acquire()
    try:
        with open(output_file_content, 'a') as outfile:
            json.dump(content_with_turns, outfile)  # Write the content with turns instead of just content
            outfile.write('\n')  # Write newline character after each response
    finally:
        lock2.release()

def read_transcript_and_analyze(input_file, output_file_full, output_file_content, startline=None, endline=None, startturn=None, endturn=None, runcount=None):
    def read_transcript_and_analyze(input_file, output_file_full, output_file_content, startline=None, endline=None, startturn=None, endturn=None, runcount=None):
    # Error checking for parameters
    if startline is not None and startturn is not None:
        raise ValueError("Parameters 'startline' and 'startturn' are mutually exclusive. Use only 'startline' to specify the line number to start analysis, or 'startturn' to specify the speaker turn number to start analysis.")
    if endline is not None and endturn is not None:
        raise ValueError("Parameters 'endline' and 'endturn' are mutually exclusive. Use only 'endline' to specify the line number to end analysis, or 'endturn' to specify the speaker turn number to end analysis.")
    if startline is not None or endline is not None or startturn is not None or endturn is not None:
        if runcount is not None:
            raise ValueError("Parameter 'runcount' cannot be used together with line or turn parameters. Use 'runcount' alone or with either 'startline' and 'endline', or 'startturn' and 'endturn'.")
    
    with open(input_file, 'r') as file:
        lines = file.readlines()

    # Process lines and extract speaker turns
    speaker_turns = process_lines(lines)
    
    # Determine start and end points for analysis
    if startline is not None:
        start = startline
    elif startturn is not None:
        start = startturn
    else:
        start = 0

    if endline is not None:
        end = endline
    elif endturn is not None:
        end = endturn
    else:
        end = len(speaker_turns)  # Updated this line to consider turns instead of lines

    # Determine number of runs
    if runcount is not None:
        end = min(start + runcount, len(speaker_turns))  # Updated this line to consider turns instead of lines

    # Check if 'end' exceeds the length of the speaker turns
    if end > len(speaker_turns):  # Updated this line to consider turns instead of lines
        end = len(speaker_turns)  # Updated this line to consider turns instead of lines
        print(f"Warning: The end point exceeded the length of the speaker turns. It has been adjusted to the last speaker turn: {end}")

    for i in range(start, end):
        # Get previous, current and next turn
        prev_turn, curr_turn, next_turn = get_turns(speaker_turns, i)
        
        # Call OpenAI API and get response
        response = prompt_standard_contextual(prev_turn, curr_turn, next_turn)
        
        # Write complete response to JSON file
        with open(output_file_full, 'a') as outfile:
            json.dump(response, outfile)
            outfile.write('\n')  # Write newline character after each response
        
        # Extract 'content' field and write to second JSON file
        content = response['choices'][0]['message']['content']
        
        # Include the turns that were passed to GPT in the content
        content_with_turns = {
            "prev_turn": prev_turn,
            "curr_turn": curr_turn,
            "next_turn": next_turn,
            "content": content
        }
        
        with open(output_file_content, 'a') as outfile:
            json.dump(content_with_turns, outfile)  # Write the content with turns instead of just content
            outfile.write('\n')  # Write newline character after each response

    # Create locks for writing to files
    lock1 = Lock()
    lock2 = Lock()

    # Use a ThreadPoolExecutor to process the lines concurrently
    with ThreadPoolExecutor(max_workers=20) as executor:
        # Assign API keys cyclically
        futures = [executor.submit(threaded_process, i, speaker_turns, output_file_full, output_file_content, lock1, lock2, open_ai_keys[i % 20]) for i in range(start, end)]
        
        # Ensure all threads have finished
        for future in as_completed(futures):
            pass
